# Volt Scripting Notebook (Complete Beginner Tutorial)

This notebook is a step-by-step guide for exploring Volt data with Python.

You do not need to be a developer to use it.

We will use:

- `voltsdk` (`VoltClient`) to talk to the Volt API
- `pandas` to organize tables
- `numpy` for math and arrays
- `matplotlib` for charts
- `scipy` for simple signal and statistics analysis

By the end, you will know how to:

- connect with your `SECRET_KEY`
- list trajectories
- list **all analyses** for a trajectory
- list exposures and timesteps
- load timestep results
- create basic charts and statistics
- download a GLB file


## Before You Start

Please edit the values in the next cell:

- `BASE_URL`: your Volt backend API URL (must include `/api`)
- `SECRET_KEY`: paste your secret key here
This notebook does **not** use environment variables. The team is resolved automatically from your secret key.


In [ ]:
BASE_URL = "http://172.20.10.5:8000/api"
SECRET_KEY = "vsk_replace_with_your_secret_key"


# Optional: if empty, the notebook will pick the first available trajectory.
TRAJECTORY_ID = ""

# Optional: if empty, the notebook will pick the first available analysis.
ANALYSIS_ID = ""

# Optional: if empty, the notebook will pick the first available exposure/timestep.
EXPOSURE_ID = ""
TIMESTEP = None


## 1. Import libraries and create the client


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, signal
from IPython.display import display

from voltsdk import VoltClient

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

client = VoltClient(
    secret_key=SECRET_KEY,
    base_url=BASE_URL,
    timeout=60,
)

print("Client is ready")
print("Base URL:", BASE_URL)


## 2. Check connectivity

The client resolves the team automatically from your secret key (using the API).

This first request also confirms your connection is working.


In [ ]:
trajectory_probe = client.list_trajectories(page=1, limit=3)
print("Connection OK")
print("Response keys:", list(trajectory_probe.keys()))
print("Preview rows:", len(trajectory_probe.get("data", [])))


## 3. List available trajectories

A trajectory is usually the main dataset or simulation run you want to inspect.


In [ ]:
trajectories_payload = client.list_trajectories(page=1, limit=100)
print("Response keys:", list(trajectories_payload.keys()))

trajectory_rows = trajectories_payload.get("data", []) if isinstance(trajectories_payload, dict) else []
trajectories_df = pd.DataFrame(trajectory_rows)

print(f"Trajectories found: {len(trajectory_rows)}")
display(trajectories_df.head(20))


## 4. Choose the current trajectory

- If `TRAJECTORY_ID` is empty, this notebook will pick the first trajectory.
- If you already know the trajectory ID, you can paste it in the configuration cell.


In [ ]:
if not TRAJECTORY_ID:
    if not trajectory_rows:
        raise ValueError("No trajectories were found for this secret key.")
    TRAJECTORY_ID = str(trajectory_rows[0].get("_id") or trajectory_rows[0].get("id"))

print("Current TRAJECTORY_ID:", TRAJECTORY_ID)


## 5. List **all analyses** for the current trajectory

This is one of the most useful steps.

It shows every analysis available for the selected trajectory.


In [ ]:
analyses_payload = client.list_analyses(trajectory_id=TRAJECTORY_ID)
print("Response keys:", list(analyses_payload.keys()))

analysis_rows = analyses_payload.get("data", []) if isinstance(analyses_payload, dict) else []
analyses_df = pd.DataFrame(analysis_rows)

print(f"Analyses found for trajectory {TRAJECTORY_ID}: {len(analysis_rows)}")
display(analyses_df.head(50))

if analyses_df.empty:
    raise ValueError("No analyses were found for this trajectory.")


## 6. Choose one analysis to explore

- If `ANALYSIS_ID` is empty, the notebook will use the first analysis.
- You can manually paste a different analysis ID if you prefer.


In [ ]:
if not ANALYSIS_ID:
    ANALYSIS_ID = str(analysis_rows[0].get("_id") or analysis_rows[0].get("id"))

print("Selected ANALYSIS_ID:", ANALYSIS_ID)


## 7. List exposures and timesteps for the selected analysis

This returns a friendly grouped result:

- exposure ID
- exposure name
- list of timesteps


In [ ]:
exposures_payload = client.list_exposures(
    trajectory_id=TRAJECTORY_ID,
    analysis_id=ANALYSIS_ID,
    limit=5000,
)

print("Response keys:", list(exposures_payload.keys()))

exposure_groups = exposures_payload.get("exposures", []) if isinstance(exposures_payload, dict) else []
exposures_df = pd.DataFrame(exposure_groups)

print(f"Exposure groups found: {len(exposure_groups)}")
display(exposures_df.head(50))


## 8. Convert exposures into one row per timestep

This makes it easier to pick a specific result to inspect.


In [ ]:
exposure_rows = []
for exposure in exposure_groups:
    exposure_id = str(exposure.get("exposureId"))
    exposure_name = exposure.get("exposureName")
    for ts in exposure.get("timesteps", []):
        exposure_rows.append({
            "analysis_id": ANALYSIS_ID,
            "exposure_id": exposure_id,
            "exposure_name": exposure_name,
            "timestep": int(ts),
        })

exposure_timestep_df = pd.DataFrame(exposure_rows)
print(f"Exposure + timestep rows: {len(exposure_timestep_df)}")
display(exposure_timestep_df.head(100))


## 9. Summary across **all analyses** in this trajectory

This step loops through every analysis and counts:

- how many exposures it has
- how many timesteps are available

This gives you a quick overview before deeper analysis.


In [ ]:
analysis_summary_rows = []

for row in analysis_rows:
    analysis_id = str(row.get("_id") or row.get("id"))
    try:
        payload = client.list_exposures(
            trajectory_id=TRAJECTORY_ID,
            analysis_id=analysis_id,
            limit=5000,
        )
        groups = payload.get("exposures", []) if isinstance(payload, dict) else []
        timestep_count = sum(len(item.get("timesteps", [])) for item in groups)
        analysis_summary_rows.append({
            "analysis_id": analysis_id,
            "analysis_name": row.get("name"),
            "exposures": len(groups),
            "timesteps": int(timestep_count),
        })
    except Exception as exc:
        analysis_summary_rows.append({
            "analysis_id": analysis_id,
            "analysis_name": row.get("name"),
            "exposures": np.nan,
            "timesteps": np.nan,
            "error": str(exc),
        })

analysis_summary_df = pd.DataFrame(analysis_summary_rows)
display(analysis_summary_df)


## 10. Select an exposure and timestep

If you leave `EXPOSURE_ID` and `TIMESTEP` empty, the notebook will choose the first available row.


In [ ]:
if exposure_timestep_df.empty:
    raise ValueError("No exposure/timestep rows were found for the selected analysis.")

if not EXPOSURE_ID or TIMESTEP is None:
    selected_row = exposure_timestep_df.iloc[0]
    EXPOSURE_ID = str(selected_row["exposure_id"])
    TIMESTEP = int(selected_row["timestep"])

print("Selected EXPOSURE_ID:", EXPOSURE_ID)
print("Selected TIMESTEP:", TIMESTEP)


## 11. Load timestep results

This fetches the data for one specific combination of:

- trajectory
- analysis
- exposure
- timestep

The result is converted into a pandas DataFrame.


In [ ]:
timestep_payload = client.get_timestep_data(
    trajectory_id=TRAJECTORY_ID,
    analysis_id=ANALYSIS_ID,
    exposure_id=EXPOSURE_ID,
    timestep=TIMESTEP,
    limit=100000,
)

print("Response keys:", list(timestep_payload.keys()))
print("Total rows:", timestep_payload.get("total"))
print("Metadata:", timestep_payload.get("_meta"))

raw_rows = timestep_payload.get("data", [])
frame_df = pd.DataFrame(raw_rows)

print(f"DataFrame shape: {frame_df.shape}")
display(frame_df.head(20))


## 12. Basic data check (beginner-friendly)

This step helps you understand what you received:

- column names
- numeric columns
- missing values


In [ ]:
if frame_df.empty:
    raise ValueError("This timestep returned no rows.")

numeric_df = frame_df.select_dtypes(include=[np.number]).copy()
numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan)

print("Columns:", list(frame_df.columns))
print("Numeric columns:", list(numeric_df.columns))
print("Missing value ratio (top 20 columns):")
display((frame_df.isna().mean().sort_values(ascending=False).head(20)).to_frame("missing_ratio"))


## 13. Quick statistics (`numpy` + `scipy`)

This gives a simple statistical summary for one numeric column:

- mean
- standard deviation
- min / max
- skewness
- kurtosis


In [ ]:
if numeric_df.empty:
    print("No numeric columns were found in this result.")
else:
    sample_col = None
    for col in numeric_df.columns:
        s = numeric_df[col].dropna()
        if len(s) >= 5:
            sample_col = col
            break
    if sample_col is None:
        sample_col = numeric_df.columns[0]

    values = numeric_df[sample_col].dropna().to_numpy(dtype=float)
    print("Sample numeric column:", sample_col)
    print("Number of values:", values.size)

    if values.size == 0:
        print("No valid numeric values after cleaning.")
    else:
        desc = stats.describe(values)
        q25, q50, q75 = np.percentile(values, [25, 50, 75])
        print("scipy.stats.describe:", desc)
        print(f"mean={np.mean(values):.6f}")
        print(f"std={np.std(values):.6f}")
        print(f"median={q50:.6f}")
        print(f"iqr={stats.iqr(values):.6f}")
        print(f"min={np.min(values):.6f}, max={np.max(values):.6f}")
        print(f"skew={stats.skew(values):.6f}, kurtosis={stats.kurtosis(values):.6f}")


## 14. Draw charts with `matplotlib`

We will draw:

- a histogram (distribution)
- a line chart (first 1000 values)


In [ ]:
if not numeric_df.empty:
    sample_col = next((c for c in numeric_df.columns if numeric_df[c].dropna().shape[0] >= 5), numeric_df.columns[0])
    values = numeric_df[sample_col].dropna().to_numpy(dtype=float)

    if values.size > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        axes[0].hist(values, bins=40, color="#2563eb", alpha=0.85)
        axes[0].set_title(f"Histogram - {sample_col}")
        axes[0].set_xlabel(sample_col)
        axes[0].set_ylabel("Count")
        axes[0].grid(alpha=0.2)

        axes[1].plot(values[:1000], color="#dc2626", linewidth=1.2)
        axes[1].set_title(f"Line chart (first 1000 values) - {sample_col}")
        axes[1].set_xlabel("Index")
        axes[1].set_ylabel(sample_col)
        axes[1].grid(alpha=0.2)

        plt.tight_layout()
        plt.show()


## 15. Simple signal analysis with `scipy.signal`

This example shows how to:

- smooth noisy values
- detect peaks

This is useful when your data behaves like a signal over time or index.


In [ ]:
if not numeric_df.empty:
    sample_col = next((c for c in numeric_df.columns if numeric_df[c].dropna().shape[0] >= 15), None)
    if sample_col is None:
        print("No numeric column has enough values for the signal example.")
    else:
        values = numeric_df[sample_col].dropna().to_numpy(dtype=float)
        if values.size < 15:
            print("Not enough values for smoothing.")
        else:
            window = min(31, values.size if values.size % 2 == 1 else values.size - 1)
            window = max(window, 5)
            if window % 2 == 0:
                window -= 1

            smooth = signal.savgol_filter(values, window_length=window, polyorder=2)
            peaks, _ = signal.find_peaks(smooth)

            print(f"Column: {sample_col}")
            print(f"Values: {values.size}, smoothing window: {window}, peaks found: {len(peaks)}")

            plt.figure(figsize=(14, 4))
            plt.plot(values[:1000], label="Raw", alpha=0.45)
            plt.plot(smooth[:1000], label="Smoothed", linewidth=2)
            peaks_in_view = peaks[peaks < 1000]
            if len(peaks_in_view) > 0:
                plt.scatter(peaks_in_view, smooth[peaks_in_view], s=18, label="Peaks")
            plt.title(f"Signal view - {sample_col}")
            plt.xlabel("Index")
            plt.ylabel(sample_col)
            plt.legend()
            plt.grid(alpha=0.2)
            plt.show()


## 16. Outlier check with z-score (`scipy.stats`)

This helps you detect unusual values.

A common rule is: values with `|z-score| > 3` may be outliers.


In [ ]:
if not numeric_df.empty:
    sample_col = next((c for c in numeric_df.columns if numeric_df[c].dropna().shape[0] >= 10), numeric_df.columns[0])
    values = numeric_df[sample_col].dropna().to_numpy(dtype=float)

    if values.size > 0:
        z = stats.zscore(values, nan_policy="omit")
        outlier_mask = np.abs(z) > 3
        outlier_count = int(np.sum(outlier_mask))

        print(f"Column: {sample_col}")
        print(f"Possible outliers (|z| > 3): {outlier_count} / {values.size}")

        preview_df = pd.DataFrame({
            sample_col: values[: min(20, values.size)],
            "zscore": z[: min(20, values.size)],
        })
        display(preview_df)


## 17. Load one sample result from **each analysis** (batch example)

This is a practical way to compare analyses.

For each analysis, this block tries to:

1. find the first available exposure
2. find the first available timestep
3. load one result
4. save a small summary row


In [ ]:
batch_rows = []

for row in analysis_rows:
    analysis_id = str(row.get("_id") or row.get("id"))
    analysis_name = row.get("name")
    try:
        exp_payload = client.list_exposures(
            trajectory_id=TRAJECTORY_ID,
            analysis_id=analysis_id,
            limit=5000,
        )
        groups = exp_payload.get("exposures", []) if isinstance(exp_payload, dict) else []
        if not groups:
            batch_rows.append({
                "analysis_id": analysis_id,
                "analysis_name": analysis_name,
                "status": "no_exposures"
            })
            continue

        first_group = groups[0]
        timesteps = first_group.get("timesteps", []) or []
        if not timesteps:
            batch_rows.append({
                "analysis_id": analysis_id,
                "analysis_name": analysis_name,
                "status": "no_timesteps"
            })
            continue

        first_exposure_id = str(first_group["exposureId"])
        first_timestep = int(timesteps[0])

        data_payload = client.get_timestep_data(
            trajectory_id=TRAJECTORY_ID,
            analysis_id=analysis_id,
            exposure_id=first_exposure_id,
            timestep=first_timestep,
            limit=50000,
        )

        rows_count = int(data_payload.get("total") or len(data_payload.get("data", [])))
        meta = data_payload.get("_meta") or {}
        per_atom_props = meta.get("properties") if isinstance(meta, dict) else None

        batch_rows.append({
            "analysis_id": analysis_id,
            "analysis_name": analysis_name,
            "status": "ok",
            "first_exposure_id": first_exposure_id,
            "first_timestep": first_timestep,
            "rows": rows_count,
            "per_atom_properties": len(per_atom_props) if isinstance(per_atom_props, list) else 0,
        })
    except Exception as exc:
        batch_rows.append({
            "analysis_id": analysis_id,
            "analysis_name": analysis_name,
            "status": "error",
            "error": str(exc),
        })

batch_results_df = pd.DataFrame(batch_rows)
display(batch_results_df)


## 18. Download the GLB file for the selected exposure/timestep

This saves the GLB file into a local `downloads/` folder.


In [ ]:
download_dir = Path("downloads")
download_dir.mkdir(parents=True, exist_ok=True)

glb_output_path = download_dir / f"{ANALYSIS_ID}-{EXPOSURE_ID}-{TIMESTEP}.glb"

saved_path = client.download_exposure_glb(
    trajectory_id=TRAJECTORY_ID,
    analysis_id=ANALYSIS_ID,
    exposure_id=EXPOSURE_ID,
    timestep=int(TIMESTEP),
    output_path=str(glb_output_path),
)

print("GLB saved to:", saved_path)
print("File size (bytes):", Path(saved_path).stat().st_size)


## 19. Next things you can try

- Change `ANALYSIS_ID`, `EXPOSURE_ID`, and `TIMESTEP` and run the workflow again
- Compare results across analyses using `batch_results_df`
- Save `frame_df` to CSV or Parquet for later use
- Build custom charts for your domain-specific metrics
